# Solution D1 · Reading a tree from the inside: nodes, paths, probabilities and rules

**Type**: Interpret

## Objective

Open a `DecisionTreeClassifier` trained on the Titanic and read its structure (`tree_`): number of nodes and leaves, root, purity of the leaves; rebuild `predict_proba` **by hand** by walking through the tree; find the leaf with the highest survival probability and write its rule in plain English.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

df = ds.titanic()
y = df["survived"].to_numpy()
X = df[["pclass", "age", "fare", "siblings_spouses", "parents_children"]].copy()
X = X.fillna(X.median())                                                   # age and fare have a few missing values
X["sex_male"] = (df["sex"] == "male").astype(int)
for t in ["Mr", "Miss", "Mrs"]:
    X[f"title_{t}"] = (df["title"] == t).astype(int)
names = list(X.columns)
X_tr, X_te, y_tr, y_te = train_test_split(X.to_numpy(), y, test_size=0.3, stratify=y, random_state=0)
tree = DecisionTreeClassifier(max_depth=4, min_samples_leaf=10, random_state=0).fit(X_tr, y_tr)
T = tree.tree_
print("columns:", names)

## Commented solution

In [ ]:
n_nodes, n_leaves, depth = T.node_count, tree.get_n_leaves(), tree.get_depth()
root, root_threshold, root_samples = names[T.feature[0]], float(T.threshold[0]), int(T.n_node_samples[0])
print(f"{n_nodes} nodes, {n_leaves} leaves, depth {depth} | root: {root} <= {root_threshold:.2f} with {root_samples} examples")

In [ ]:
def proba_by_hand(tree, x):
    T = tree.tree_
    n = 0
    while T.children_left[n] != -1:                                       # until you are in a leaf
        n = T.children_left[n] if x[T.feature[n]] <= T.threshold[n] else T.children_right[n]
    v = T.value[n][0]
    return v / v.sum()                                                     # normalize: scikit-learn versions differ (counts or fractions)

In [ ]:
by_hand = np.array([proba_by_hand(tree, x) for x in X_te])
matches = bool(np.allclose(by_hand, tree.predict_proba(X_te)))
print("proba by hand = predict_proba on all the", len(X_te), "test examples:", matches)

In [ ]:
leaves = np.flatnonzero(T.children_left == -1)
samples = T.n_node_samples[leaves]
purities = np.array([T.value[n][0].max() / T.value[n][0].sum() for n in leaves])
largest_leaf = int(leaves[samples.argmax()])
largest_purity = float(purities[samples.argmax()])
n_pure_leaves = int(np.isclose(purities, 1.0).sum())
print(f"largest leaf: node {largest_leaf} with {samples.max()} examples and purity {largest_purity:.2f} | pure leaves: {n_pure_leaves} out of {len(leaves)}")

In [ ]:
parent = {}
for n in range(T.node_count):
    if T.children_left[n] != -1:
        parent[T.children_left[n]] = (n, "<=")
        parent[T.children_right[n]] = (n, ">")
large = [n for n in leaves if T.n_node_samples[n] >= 20]
best_leaf = max(large, key=lambda n: T.value[n][0][1] / T.value[n][0].sum())              # probability of class 1 (survived)
rule, n = [], best_leaf
while n in parent:
    p, sign = parent[n]
    rule.append(f"{names[T.feature[p]]} {sign} {T.threshold[p]:.1f}")
    n = p
rule = rule[::-1]
rule_proba = float(T.value[best_leaf][0][1] / T.value[best_leaf][0].sum())
print(f"the safest leaf (>= 20 examples, node {best_leaf}, {T.n_node_samples[best_leaf]} examples): {rule_proba:.0%} survive if")
for r in rule:
    print("   ", r)

## Check

In [ ]:
with verify():
    check(n_nodes == T.node_count and n_leaves == tree.get_n_leaves() and depth == tree.get_depth() and n_nodes == 2 * n_leaves - 1, "nodes, leaves and depth read from tree_ (a binary tree has 2 * leaves - 1 nodes)", "T.node_count, get_n_leaves(), get_depth()")
    check(root == names[T.feature[0]] and close(root_threshold, T.threshold[0]) and root_samples == len(y_tr), "root, threshold and samples at the root (the whole training set)", "T.feature[0], T.threshold[0], T.n_node_samples[0]")
    check(matches is True, "proba_by_hand matches predict_proba on the whole test set", "go down with children_left/right; normalize tree_.value")
    leaves = np.flatnonzero(T.children_left == -1)
    samples = T.n_node_samples[leaves]
    check(largest_leaf == leaves[samples.argmax()] and samples.max() >= 100 and largest_purity > 0.7, "the largest leaf has over 100 examples (and purity above 0.7)", "n_node_samples of the leaves")
    purities = np.array([T.value[n][0].max() / T.value[n][0].sum() for n in leaves])
    check(n_pure_leaves == int((np.isclose(purities, 1.0)).sum()) and n_pure_leaves >= 1, "correct number of pure leaves (at least one)", "purity = max(value) / sum(value)")
    check(isinstance(rule, list) and len(rule) >= 3 and all(isinstance(r, str) and ("<=" in r or ">" in r) for r in rule) and rule_proba > 0.85, "rule with at least 3 conditions and survival probability over 0.85", "climb with a child -> parent dictionary")
    check(any("sex_male" in r or "title_Mr" in r for r in rule), "the rule involves sex (or title)", "the conditions of the safest leaf")
done()

## What you learned

- **A tree is a table**: `tree_` holds, for each node, the feature and the threshold, the two children and the counts. `predict_proba` is a descent from one node to the next followed by a normalization; the rebuild by hand matches on every example. There is nothing more to it.
- **Binary trees have `2 × leaves − 1` nodes**: the leaves tell the complexity of the model and the depth the maximum length of an explanation.
- **The purity of a leaf is its uncertainty:** with `min_samples_leaf=10` many leaves are not pure and the probability is a fraction over at least ten examples. With leaves of 1-2 examples the same "probability" would be 0 or 1 and meaningless.
- **Every prediction has a written rule**: the safest leaf among those with at least 20 examples is a combination of a few conditions (women and children in the first classes): climbing the tree you produce the explanation "this person survives with a probability of 97% because...". It is the advantage that no other model in the course has.

## Common pitfall

Reading `tree_.value` without checking the version: in scikit-learn < 1.4 it holds the **counts**, from 1.4 the fractions. Code that uses the counts as fractions (or the other way round) breaks silently when the library is updated: always normalize.

## Interview question

<details><summary>How do you explain the prediction of a tree to a customer?</summary>

I show the path from the example to the leaf (`decision_path`) as a list of conditions ("male, third class, age above 9"), and the fraction of similar training examples that are in the leaf, with their number: it is an exact local explanation, not an approximation.
With deep trees or forests the explanation becomes more complex and I use permutation importances or SHAP.

</details>